In [5]:
import cv2, time, numpy as np
from ultralytics import YOLO
import mediapipe as mp
from fastdtw import fastdtw
from scipy.spatial.distance import cdist
from tqdm import tqdm

# -------------------------------------------------------
#  Model loading
# -------------------------------------------------------
model_yolo_default = YOLO("NeuroYolo\yolo11n-pose.pt")  # default
model_neuroyolo = YOLO(f'best.pt') 
mp_pose = mp.solutions.pose.Pose(static_image_mode=False, model_complexity=1)

In [2]:
# -------------------------------------------------------
#  Inference wrappers
# -------------------------------------------------------

def infer_yolo(model, frame):
    results = model.predict(frame, verbose=False)
    keypoints = results[0].keypoints.xy.cpu().numpy() if results[0].keypoints is not None else np.zeros((1,17,2))
    return keypoints

def infer_mediapipe(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = mp_pose.process(rgb)
    if not res.pose_landmarks:
        return np.zeros((1,33,2))
    kp = np.array([[lm.x*frame.shape[1], lm.y*frame.shape[0]] for lm in res.pose_landmarks.landmark])
    return kp[None, :, :17] if kp.shape[0] >= 17 else np.zeros((1,17,2))

# -------------------------------------------------------
#  Metric functions (same as before)
# -------------------------------------------------------

def compute_jitter_variance(keypoints):
    if len(keypoints) < 2:
        return np.nan
    diffs = np.diff(keypoints, axis=0)
    return np.var(np.linalg.norm(diffs, axis=2))

def compute_msi(keypoints):
    if keypoints.shape[1] < 3: return np.nan
    A, B, C = keypoints[:,5,:], keypoints[:,6,:], keypoints[:,7,:]
    v1, v2 = A-B, C-B
    cos = np.sum(v1*v2,axis=1)/(np.linalg.norm(v1,axis=1)*np.linalg.norm(v2,axis=1)+1e-6)
    ang = np.degrees(np.arccos(np.clip(cos,-1,1)))
    return np.mean(np.abs(np.diff(ang)))

def align_keypoints_dim(kp, target_dim=17):
    """Ensures both models output the same joint dimensionality."""
    if kp.shape[1] == target_dim:
        return kp
    elif kp.shape[1] > target_dim:
        # keep first 17 joints (COCO-compatible subset)
        return kp[:, :target_dim, :]
    else:
        # pad with zeros if fewer
        pad = np.zeros((kp.shape[0], target_dim - kp.shape[1], 2))
        return np.concatenate([kp, pad], axis=1)
    
def normalize_pose(kp):
    """Normalize by torso length to remove scale bias."""
    if kp.size == 0:
        return kp
    # shoulders and hips indices for COCO (5,6,11,12)
    torso = np.linalg.norm(kp[:,5,:] - kp[:,11,:], axis=1).mean() + 1e-6
    return kp / torso

def compute_psm(kp1, kp2):
    """ Pose Similarity Metric using DTW between flattened joint coordinates """
    # add before DTW
    kp1 = normalize_pose(align_keypoints_dim(kp1))
    kp2 = normalize_pose(align_keypoints_dim(kp2))

    kp1 = align_keypoints_dim(kp1)
    kp2 = align_keypoints_dim(kp2)
    n = min(len(kp1), len(kp2))
    s1, s2 = kp1[:n].reshape(n, -1), kp2[:n].reshape(n, -1)
    dist, _ = fastdtw(s1, s2)
    return max(0, 100 - dist / max(len(s1), len(s2)))

# -------------------------------------------------------
#  Evaluation routine
# -------------------------------------------------------

def run_video(video_path, model_func, label):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(frame)
    cap.release()

    # inference & timing
    t0 = time.time()
    all_kp = []
    for f in tqdm(frames, desc=label):
        kp = model_func(f)
        all_kp.append(kp)
    avg_ms = (time.time()-t0)/len(frames)*1000
    return np.vstack(all_kp), avg_ms

In [3]:
if __name__ == "__main__":

    video_participant = "videos\C1_6.mp4"
    video_coach = "videos\P1_6.mp4"

    # Run models
    kp_mediapipe, t_mp = run_video(video_participant, infer_mediapipe, "MediaPipe")
    kp_yolo_def, t_def = run_video(video_participant, lambda f: infer_yolo(model_yolo_default,f), "YOLOv11-default")
    kp_neuro, t_ft = run_video(video_participant, lambda f: infer_yolo(model_neuroyolo,f), "NeuroYOLO-finetuned")

    # Reference
    kp_coach, _ = run_video(video_coach, lambda f: infer_yolo(model_neuroyolo,f), "Coach Reference")

    # Metrics
    results = []
    for name, kp, t in [("MediaPipe", kp_mediapipe, t_mp),
                        ("YOLOv11-default", kp_yolo_def, t_def),
                        ("NeuroYOLO-finetuned", kp_neuro, t_ft)]:
        jitter = compute_jitter_variance(kp)
        msi = compute_msi(kp)
        psm = compute_psm(kp, kp_coach)
        results.append((name, t, jitter, msi, psm))

    print("\n--- Comparison Results ---")
    print(f"{'Model':<22}{'ms/frame':>10}{'Jitter Var':>15}{'MSI':>10}{'PSM':>10}")
    for name, t, j, m, p in results:
        print(f"{name:<22}{t:>10.2f}{j:>15.3f}{m:>10.2f}{p:>10.2f}")


Coach Reference: 100%|██████████| 1800/1800 [00:33<00:00, 53.67it/s]



--- Comparison Results ---
Model                   ms/frame     Jitter Var       MSI       PSM
MediaPipe                  24.56         12.772      0.61     57.66
YOLOv11-default            20.17         44.055      0.34     61.38
NeuroYOLO-finetuned        17.57          4.311      0.13     57.86
